In [0]:
import json
import os
from urllib.parse import urljoin

import pandas as pd
import requests

BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
from pprint import pprint

In [0]:
# 1/18

def get_json(url: str) -> dict:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.json()


In [0]:
# 2/18

def fetch_all(first_url: str) -> list:
    orders = [] 
    url = first_url
    while url:
        page = get_json(url)
        orders.extend(page['data'])
        url = urljoin(url, page['next']) if page['next'] else None
    return orders

# orders = fetch_all(BASE_URL + 'orders/page-1.json')

In [0]:
# 3/18

def save_json(pages: list, path: str) -> None:
    with open(path, 'w') as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)

# save_json(orders, os.path.join(RAW_DIR, 'orders.json'))

In [0]:
# 4/18

def download_file(url: str, path: str) -> int:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(path, 'wb') as f:
        f.write(response.content)
    return len(response.content)

# download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR, 'teams.csv'))
# download_file((FILES_URL + 'products.csv'), os.path.join(RAW_DIR, 'products.csv'))
# download_file((FILES_URL + 'matches.parquet'), os.path.join(RAW_DIR, 'matches.parquet'))

In [0]:
# 5/18

def extract(raw_dir: str) -> dict:
    
    download_file(FILES_URL + 'teams.csv', os.path.join(raw_dir, 'teams.csv'))
    download_file(FILES_URL + 'products.csv', os.path.join(raw_dir, 'products.csv'))
    download_file(FILES_URL + 'matches.parquet', os.path.join(raw_dir, 'matches.parquet'))
    
    orders = fetch_all(BASE_URL + 'orders/page-1.json')
    save_json(orders, os.path.join(raw_dir, 'orders.json'))

    teams_df = pd.read_csv(raw_dir + '/' + 'teams.csv')
    products_df = pd.read_csv(raw_dir + '/' + 'products.csv')
    matches_df = pd.read_parquet(raw_dir + '/' +  'matches.parquet')

    orders_df = pd.json_normalize(orders, sep='_').rename(columns = {'customer_name': 'customer', 'customer_city': 'city'})

    return {'orders': orders_df, 'products': products_df, 'teams': teams_df, 'matches': matches_df}

# display(extract(RAW_DIR)['products'])
products = extract(RAW_DIR)['products']
orders = extract(RAW_DIR)['orders']

In [0]:
# 6/18

def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    is_dup = df.duplicated('product_id', keep='first')
    df = df[~is_dup]
    df['team_code'] = df['team_code'].str.strip().str.upper()
    df['category'] = df['category'].str.strip().str.lower()

    df['price'] = df['price'].str.replace('RSD', '', regex = False).str.replace(',', '.').str.strip()
    df['price'] = pd.to_numeric(df['price'])

    d = {'yes': True, 'Yes': True, 'YES': True}
    df['active'] = df['active'].map(d).fillna(False)

    return df

product_ids = set(clean_products(products)['product_id'])


/home/spark-511207ec-4a56-40ba-9abb-9c/.ipykernel/74/command-5414494493700922-3358397700:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['active'] = df['active'].map(d).fillna(False)


In [0]:
# 7/18

def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    for column in ['order_id', 'customer', 'city', 'payment']:
        df[column] = df[column].str.strip()

    df['order_id'] = df['order_id'].str.upper()
    df['customer'] = df['customer'].str.title()
    df['city'] = df['city'].str.title().fillna('Unknown')

    return df

# with pd.option_context("display.max_rows", 1000):
#     display(clean_orders(orders)[['order_id', 'customer', 'city', 'payment', 'quantity']])


order_id,customer,city,payment,quantity
NSS-00001,Nikola Stojanović,Zrenjanin,cash,1
NSS-00002,Jovana Pavlović,Čačak,card,3
NSS-00003,Sara Živković,Beograd,card,1
NSS-00004,Sara Đorđević,Subotica,cash,1
NSS-00005,Stefan Popović,Niš,card,1
NSS-00006,Jovana Đorđević,Kraljevo,cash,3
NSS-00007,Petar Ilić,Sombor,card,2
NSS-00008,Nikola Jovanović,Kraljevo,cash,3
NSS-00009,Milica Ilić,Kragujevac,cash,3
NSS-00010,Ana Kovačević,Beograd,paypal,2


In [0]:
# 8/18

def remove_duplicate_orders(df: pd.DataFrame) -> tuple:
    is_dup = df.duplicated('order_id', keep='first')
    return (df[~is_dup], df[is_dup])

# remove_duplicate_orders(orders)

In [0]:
# 9/18

def validate_order(row: dict, product_ids: set) -> str:
    quantity = pd.to_numeric(row['quantity'], errors='coerce')
    unit_price = pd.to_numeric(row['unit_price'], errors='coerce')
    
    if not row['order_id'].startswith('NSS-'):
        return 'invalid_order'
    if pd.isna(row['quantity']):
        return 'missing_quantity'
    if pd.isna(quantity) or quantity <= 0 or quantity != int(quantity):
        return 'invalid_quantity'
    if pd.isna(unit_price) or unit_price <= 0:
        return 'invalid_price'
    if row['product_id'] not in product_ids:
        return 'unknown_product'
    
    return 'ok'

orders = clean_orders(orders)
# for i in range(len(orders)):
#     print(orders.iloc[i]['order_id'], validate_order(orders.iloc[i], product_ids))

# orders.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1852 entries, 0 to 1851
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   order_id    1852 non-null   object
 1   order_date  1852 non-null   object
 2   product_id  1852 non-null   int64 
 3   quantity    1844 non-null   object
 4   unit_price  1849 non-null   object
 5   payment     1852 non-null   object
 6   customer    1852 non-null   object
 7   city        1852 non-null   object
dtypes: int64(1), object(7)
memory usage: 115.9+ KB


In [0]:
# 10/18

def transform_orders(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['quantity'] = pd.to_numeric(df['quantity'], errors = 'coerce', downcast='int64')
    df['unit_price'] = pd.to_numeric(df['unit_price'], errors = 'coerce', downcast='float')
    df['order_date'] = pd.to_datetime(df['order_date'])
    df['total'] = df['quantity'] * df['unit_price']

    return df

transform_orders(orders).info()

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-7469565379815287>, line 13
      9     df['total'] = df['quantity'] * df['unit_price']
     11     return df
---> 13 transform_orders(orders).info()

File <command-7469565379815287>, line 6, in transform_orders(df)
      3 def transform_orders(df: pd.DataFrame) -> pd.DataFrame:
      4     df = df.copy()
----> 6     df['quantity'] = pd.to_numeric(df['quantity'], errors = 'coerce', downcast='int')
      7     df['unit_price'] = pd.to_numeric(df['unit_price'], errors = 'coerce', downcast='float')
      8     df['order_date'] = pd.to_datetime(df['order_date'])

File /databricks/python/lib/python3.12/site-packages/pandas/core/tools/numeric.py:170, in to_numeric(arg, errors, downcast, dtype_backend)
     53 """
     54 Convert argument to a numeric type.
     55 
   (...)    167 dtype: Float32
    168 """
    169 if downcast no

In [0]:
# def load(outputs: dict, processed_dir: str) -> dict:
#     written = {}
#     for name, df in outputs.items():
#         path = os.path.join(processed_dir, name)
#         if name.endswith('.parquet'):
#             df.to_parquet(path, index=False)
#         else:
#             df.to_csv(path, index=False)
#         written[name] = len(df)
#     return written

In [0]:
# def run_pipeline(raw_dir: str, processed_dir: str) -> dict:
#     data = extract(raw_dir)
#     products = clean_products(data['products'])
#     valid, rejected = split_orders(data['orders'], products)
#     results = team_results(data['matches'], data['teams'])
#     windows = match_windows(results)
#     outputs = {
#         'clean_orders.parquet': valid,
#         'rejected_orders.csv': rejected,
#         'products.csv': products,
#         'league_table.csv': league_table(results),
#         'revenue_by_team.csv': revenue_by(valid.dropna(subset=['team_code']), 'team_code'),
#         'revenue_by_category.csv': revenue_by(valid, 'category'),
#         'sales_after_results.csv': sales_after_results(valid, windows),
#         }
#     written = load(outputs, processed_dir)
#     summary = {
#         'total_rows': len(data['orders']),
#         'duplicates': int((rejected['reason'] == 'duplicate').sum()),
#         'valid_rows': len(valid),
#         'rejected_rows': len(rejected),
#         'total_revenue': round(float(valid['total'].sum()), 2),
#         'files_written': len(written) + 1,
#         }
#     pd.DataFrame(list(summary.items()), columns=['metric',
#     'value']).to_csv(
#         os.path.join(processed_dir, 'summary.csv'), index=False)
#     return summary